In [1]:
from src.adapter_training.adapter_prompt import ADAPTER_SYSTEM_PROMPT

In [2]:
import pandas as pd

df = pd.read_pickle('./data/full_df_deduped_fixed_tool_calls.pkl')
df.head()

,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type,corrupted_response
0,NousResearch/hermes-function-calling-v1,func_calling,977,2e583020-ea45-4496-aafe-a6d141ed1ad7,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'an...",Consumer Staples Software,Drug Retail Supply Chain Management,Streamline supply chain operations for drug re...,"[{'role': 'system', 'content': 'You are a func...",corrupt_tool_call_name,"<tool_call>\n{""arguments"": {""product_database""..."
1,NousResearch/hermes-function-calling-v1,func_calling,1587,99abd602-3854-41e1-9fe7-31dcef70ad39,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
2,NousResearch/hermes-function-calling-v1,func_calling,1360,beb07363-1986-4701-a4b7-27e04536f093,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",delete_content,
3,NousResearch/hermes-function-calling-v1,func_calling,1738,28558c88-cfd6-431e-894a-3d7510b416ba,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",no_corruption,"<tool_call>\n{""arguments"": {""queries"": ['Can y..."
4,NousResearch/hermes-function-calling-v1,func_calling,1347,248b82c3-e473-40e8-91ff-74d9b314cd1f,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",add_extra_tool_call,"<tool_call>\n{""arguments"": {""queries"": ['What ..."


In [3]:
tmp_df = df[df['corruption_type'] == 'add_extra_tool_call']
tmp_df.head()

,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type,corrupted_response
4,NousResearch/hermes-function-calling-v1,func_calling,1347,248b82c3-e473-40e8-91ff-74d9b314cd1f,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",add_extra_tool_call,"<tool_call>\n{""arguments"": {""queries"": ['What ..."
5,NousResearch/hermes-function-calling-v1,func_calling,1426,f5460613-ebcb-4bee-b55e-e6b3e80cdfa1,"[{'from': 'system', 'value': 'You are an exper...","[{'type': 'function', 'function': {'name': 'Ex...",Information Extraction,Json Schema,Structured json schema extaction with function...,"[{'role': 'system', 'content': 'You are an exp...",add_extra_tool_call,"<tool_call>\n{""arguments"": {""queries"": ['How c..."
20,NousResearch/hermes-function-calling-v1,func_calling,965,f31dfa0f-e47f-4cb2-b41f-220ce99f24ec,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'cr...",SaaS Platforms,Zapier,Integration,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call,The integration between Salesforce CRM and Tre...
21,NousResearch/hermes-function-calling-v1,func_calling,330,3d22780a-8cf3-4a8f-abb1-3c7783b9ea31,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'es...",Utilities Software,Power Producers Remote Monitoring System,Develop a remote monitoring system for power p...,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call,The secure connections to the wind turbine ass...
27,NousResearch/hermes-function-calling-v1,func_calling,808,7da227ad-771b-4627-aa0f-bd28b8dc4344,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",Financial Software,Banking Customer Relationship Management (CRM),Manage customer relationships for banks.,"[{'role': 'system', 'content': 'You are a func...",add_extra_tool_call,Here are the outcomes of the analyses conducte...


In [4]:
idx = 20
conversation = tmp_df.loc[idx, 'messages']
corrupted_response = tmp_df.loc[idx, 'corrupted_response']
print(conversation)
print('---')
print(corrupted_response)
print('---')
print(len(conversation))

[{'role': 'system', 'content': "You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions."}, {'role': 'user', 'content': "I am currently overseeing a project that requires efficient coordination between our Customer Relationship Management (CRM) system and our project management tool. We utilize Salesforce as our CRM platform, where we register new sales leads. The requirement is to automate the creation of tasks in our project management tool, Trello, corresponding to each new lead registered.\n\nFor every new lead captured in Salesforce, I need a task created in Trello that will contain the lead's full name, their email address, and the specific product they are interested in. This automation is crucial for prompt follow-ups and ensuring no opportunities are missed. \n\nCould you please set up this in

In [21]:
# build adapter prompt
import json
from typing import Any

def build_adapter_draft_payload(
    content: str, tool_calls: list[dict[str, Any]] | None
) -> str:
    if tool_calls is not None: 
        tool_calls_json = json.dumps(tool_calls or [], sort_keys=True)
        tool_calls_json = [f"<tool_call>\n{tool_call.strip()}\n</tool_call>" for tool_call in tool_calls_json]
        tool_calls_json = "\n".join(tool_calls_json)
    else:
        tool_calls_json = None
    return '\n'.join((
        "<DRAFT>",
        f"{content}",
        f"{tool_calls_json}" if tool_calls_json is not None else "",
        "</DRAFT>"
    ))


draft = build_adapter_draft_payload(corrupted_response, None)
print(draft)

<DRAFT>
The integration between Salesforce CRM and Trello has been successfully set up. Now every new lead registered in Salesforce will automatically generate a task in Trello containing the lead's full name, their email address, and the specific product they are interested in.

Here are the details of the integration:
- CRM System: Salesforce
- Project Management Tool: Trello
- Fields Mapped:
  - Lead's Full Name
  - Email Address
  - Interested Product

The integration is operational, streamlining your lead management process. Should you ever need to change settings, check the status, or remove the integration, please remember the Integration ID: 'integ_54321'.
<tool_call>
{"arguments": {"integration_id": "integ_54321"}, "name": "get_integration_status"}
</tool_call>

</DRAFT>


In [6]:
# conversation history for adapter

conversation

[{'role': 'system',
  'content': "You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions."},
 {'role': 'user',
  'content': "I am currently overseeing a project that requires efficient coordination between our Customer Relationship Management (CRM) system and our project management tool. We utilize Salesforce as our CRM platform, where we register new sales leads. The requirement is to automate the creation of tasks in our project management tool, Trello, corresponding to each new lead registered.\n\nFor every new lead captured in Salesforce, I need a task created in Trello that will contain the lead's full name, their email address, and the specific product they are interested in. This automation is crucial for prompt follow-ups and ensuring no opportunities are missed. \n\nCould you please set up th

In [7]:
# assume you will always have a conversation system prompt
conversation_system_prompt = conversation[0]['content']

In [22]:
# first is system prompt, last is draft message
def get_conversation_history(conversation):
    conversation_history = [f'[{x["role"].upper()}]: {x["content"]}' for x in conversation[1:-1]]
    conversation_history = "\n---\n".join(conversation_history)
    return conversation_history

conversation_history = get_conversation_history(conversation)
print(conversation_history)


[USER]: I am currently overseeing a project that requires efficient coordination between our Customer Relationship Management (CRM) system and our project management tool. We utilize Salesforce as our CRM platform, where we register new sales leads. The requirement is to automate the creation of tasks in our project management tool, Trello, corresponding to each new lead registered.

For every new lead captured in Salesforce, I need a task created in Trello that will contain the lead's full name, their email address, and the specific product they are interested in. This automation is crucial for prompt follow-ups and ensuring no opportunities are missed. 

Could you please set up this integration using Zapier? The following details can be used for the function call:

1. CRM system: Salesforce
2. Project management tool: Trello
3. Required fields for task creation: Lead's full name, email address, and interested product

Additionally, based on the successful function call result previou

In [23]:
user_msg_template = """### Conversation system prompt:
{conversation_system_prompt}

===

### Conversation history:
{conversation_history}

===

### Draft:
{draft}
"""

user_msg = user_msg_template.format(
    conversation_system_prompt=conversation_system_prompt,
    conversation_history=conversation_history,
    draft=draft
)
print(user_msg)

### Conversation system prompt:
You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.

===

### Conversation history:
[USER]: I am currently overseeing a project that requires efficient coordination between our Customer Relationship Management (CRM) system and our project management tool. We utilize Salesforce as our CRM platform, where we register new sales leads. The requirement is to automate the creation of tasks in our project management tool, Trello, corresponding to each new lead registered.

For every new lead captured in Salesforce, I need a task created in Trello that will contain the lead's full name, their email address, and the specific product they are interested in. This automation is crucial for prompt follow-ups and ensuring no opportunities are missed. 

Could you please set up thi

In [11]:
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal, Any


class AdapterPatch(BaseModel):
    model_config = ConfigDict(extra="forbid")

    op: Literal["add", "replace", "remove"]
    path: str
    value: Any


class AdapterStructuredOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    decision: Literal["lgtm", "patch"]
    patches: list[AdapterPatch] = Field(default_factory=list)


In [12]:
AdapterStructuredOutput.model_json_schema()

{'$defs': {'AdapterPatch': {'additionalProperties': False,
   'properties': {'op': {'enum': ['add', 'replace', 'remove'],
     'title': 'Op',
     'type': 'string'},
    'path': {'title': 'Path', 'type': 'string'},
    'value': {'title': 'Value'}},
   'required': ['op', 'path', 'value'],
   'title': 'AdapterPatch',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'decision': {'enum': ['lgtm', 'patch'],
   'title': 'Decision',
   'type': 'string'},
  'patches': {'items': {'$ref': '#/$defs/AdapterPatch'},
   'title': 'Patches',
   'type': 'array'}},
 'required': ['decision'],
 'title': 'AdapterStructuredOutput',
 'type': 'object'}

In [13]:
gold_response = conversation[-1]['content'].strip()
print(gold_response)

The integration between Salesforce CRM and Trello has been successfully set up. Now every new lead registered in Salesforce will automatically generate a task in Trello containing the lead's full name, their email address, and the specific product they are interested in.

Here are the details of the integration:
- CRM System: Salesforce
- Project Management Tool: Trello
- Fields Mapped:
  - Lead's Full Name
  - Email Address
  - Interested Product

The integration is operational, streamlining your lead management process. Should you ever need to change settings, check the status, or remove the integration, please remember the Integration ID: 'integ_54321'.


In [14]:
print(corrupted_response)

The integration between Salesforce CRM and Trello has been successfully set up. Now every new lead registered in Salesforce will automatically generate a task in Trello containing the lead's full name, their email address, and the specific product they are interested in.

Here are the details of the integration:
- CRM System: Salesforce
- Project Management Tool: Trello
- Fields Mapped:
  - Lead's Full Name
  - Email Address
  - Interested Product

The integration is operational, streamlining your lead management process. Should you ever need to change settings, check the status, or remove the integration, please remember the Integration ID: 'integ_54321'.
<tool_call>
{"arguments": {"integration_id": "integ_54321"}, "name": "get_integration_status"}
</tool_call>


In [24]:
# data generation

data_generation_prompt_template = f"""
===

i need you to generate a patch for the above draft, so that the response looks like below gold response.

Gold response:
{gold_response}
"""

modified_user_msg = user_msg + data_generation_prompt_template.format(gold_response=gold_response)
print(modified_user_msg)

### Conversation system prompt:
You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.

===

### Conversation history:
[USER]: I am currently overseeing a project that requires efficient coordination between our Customer Relationship Management (CRM) system and our project management tool. We utilize Salesforce as our CRM platform, where we register new sales leads. The requirement is to automate the creation of tasks in our project management tool, Trello, corresponding to each new lead registered.

For every new lead captured in Salesforce, I need a task created in Trello that will contain the lead's full name, their email address, and the specific product they are interested in. This automation is crucial for prompt follow-ups and ensuring no opportunities are missed. 

Could you please set up thi

In [19]:
from openai import AsyncOpenAI
client = AsyncOpenAI(base_url="http://localhost:8000/v1",api_key="")

responses = await client.chat.completions.parse(
    model="glm-4.7-fp8",
    messages=[
        {'role': 'system', 'content': ADAPTER_SYSTEM_PROMPT},
        {"role": "user", "content": modified_user_msg}
    ],
    response_format=AdapterStructuredOutput,
    max_tokens=32768,
)

parsed_response = responses.choices[0].message.parsed
print(parsed_response)

decision='patch' patches=[AdapterPatch(op='remove', path='/tool_calls', value='')]


In [20]:
print(parsed_response.model_dump_json())

{"decision":"patch","patches":[{"op":"remove","path":"/tool_calls","value":""}]}


In [27]:
from pydantic import ValidationError

async def generate_patch(conversation, corrupted_response):
    conversation_system_prompt = conversation[0]['content']
    conversation_history = get_conversation_history(conversation)
    draft = build_adapter_draft_payload(corrupted_response, None)
    user_msg = user_msg_template.format(
        conversation_system_prompt=conversation_system_prompt,
        conversation_history=conversation_history,
        draft=draft)
    gold_response = conversation[-1]['content'].strip()
    modified_user_msg = user_msg + data_generation_prompt_template.format(gold_response=gold_response)
    for _ in range(3):
        try:
            responses = await client.chat.completions.parse(
                model="glm-4.7-fp8",
                messages=[
                    {'role': 'system', 'content': ADAPTER_SYSTEM_PROMPT},
                    {"role": "user", "content": modified_user_msg}
                ],
                response_format=AdapterStructuredOutput,
                max_tokens=32768,
            )
            parsed_response = responses.choices[0].message.parsed
            return parsed_response
        except ValidationError as e:
            continue
    return None


patch = await generate_patch(conversation, corrupted_response)
print(patch)


decision='patch' patches=[AdapterPatch(op='remove', path='/tool_calls', value='')]


In [36]:
import asyncio
from tqdm import tqdm

tmp_df = df.sample(50)

semaphore = asyncio.Semaphore(32)

async def limited_generate_patch(conversation, corrupted_response, corruption_type, pbar):
    async with semaphore:
        if corruption_type == 'no_corruption':
            patch = "{'decision': 'lgtm', 'patches': []}"
        else:
            patch = await generate_patch(conversation, corrupted_response)
            patch = patch.model_dump_json()
        pbar.update(1)
        return patch

pbar = tqdm(total=len(tmp_df), ncols=80)
tasks = []
for _, row in tmp_df.iterrows():
    conversation = row['messages']
    corrupted_response = row['corrupted_response']
    corruption_type = row['corruption_type']
    tasks.append(limited_generate_patch(conversation, corrupted_response, corruption_type, pbar))

patches = await asyncio.gather(*tasks, return_exceptions=True)

 98%|██████████████████████████████████████████▏| 49/50 [00:45<00:01,  1.90s/it]

In [37]:
pbar.close()

 98%|██████████████████████████████████████████▏| 49/50 [00:45<00:00,  1.08it/s]


In [38]:
len(patches)


50

In [39]:
patches

['{"decision":"patch","patches":[{"op":"replace","path":"/content","value":"The integration between Salesforce CRM and Trello has been successfully set up. Now every new lead registered in Salesforce will automatically generate a task in Trello containing the lead\'s full name, their email address, and the specific product they are interested in.\\n\\nHere are the details of the integration:\\n- CRM System: Salesforce\\n- Project Management Tool: Trello\\n- Fields Mapped:\\n  - Lead\'s Full Name\\n  - Email Address\\n  - Interested Product\\n\\nThe integration is operational, streamlining your lead management process. Should you ever need to change settings, check the status, or remove the integration, please remember the Integration ID: \'integ_54321\'."},{"op":"remove","path":"/tool_calls/3","value":""},{"op":"remove","path":"/tool_calls/2","value":""},{"op":"remove","path":"/tool_calls/1","value":""},{"op":"remove","path":"/tool_calls/0","value":""}]}',
 "{'decision': 'lgtm', 'patch